## Import packages

In [ ]:
import astropy
from pylab import *
from astropy.io import fits
from astropy.time import Time
from PyAstronomy import pyasl
from astropy import units as u
from astropy.coordinates import SkyCoord, EarthLocation

## Reading the header

In [ ]:
# AF's line is `hdul = fits.open('<star>blue_multi.fits'); hdul[0].header`.
# We were not given the multi file — only the header block she links for HE 1158-2313, which is
# 5 FITS blocks with no data and no END card, so fits.open() raises. Same idea, read as text.
import re
RAW = "../../../data/he1158-2313blue_multi_HEADERONLY.fits"
raw = open(RAW, "rb").read().decode("latin-1")
header = {k: v.strip().strip("'").strip() for k, v in
          re.findall(r"([A-Z0-9_-]{1,8})\s*=\s*('[^']*'|[^/\n]{1,40})", raw)}
for k in ("OBJECT", "UT-DATE", "UT-START", "UT-TIME", "RA", "DEC", "EQUINOX",
          "SITENAME", "ORIGIN", "SITELAT", "SITELONG", "SITEALT", "INSTRUME", "EXPTIME", "AIRMASS"):
    if k in header:
        print(f"{k:10s} = {header[k]}")

### Save for the other notebooks

In [ ]:
import json, os
os.makedirs("results", exist_ok=True)
keep = {k: header[k] for k in ("OBJECT","UT-DATE","UT-START","RA","DEC","EQUINOX",
                               "SITENAME","ORIGIN","SITELAT","SITELONG","SITEALT",
                               "INSTRUME","EXPTIME","AIRMASS") if k in header}
keep["_source"] = RAW
keep["_note"]   = "AF's header_information.ipynb, adapted: header block parsed as text (no END card)"
# no HELIO/BARY keyword exists in this header - record that explicitly, it matters for 4.4
# NB: AF's `from pylab import *` shadows the builtin `any` with `numpy.any`, and
# np.any(<generator>) is True for any generator - so pass a list, not a generator.
_vk = ("HELIO", "BARY", "VCOR", "VHEL")
keep["_velocity_keywords"] = sorted([k for k in header if any([s in k.upper() for s in _vk])])
json.dump(keep, open("results/part4-4_header.json","w"), indent=2)
print(json.dumps(keep, indent=2))